Stanley-Reisner-Attention

In [ ]:
'''
Attention pattern (directed graph)
        ↓
Threshold + symmetrization → Simplicial complex Δ
        ↓
Face ring S/I_Δ ←──→ Minimal free resolution (Betti table)
        ↓                    ↓
   Algebraic geometry    Homotopy type |Δ|
        ↓                    ↓
   Toric variety       Sheaf cohomology H^i(P(Δ); ℱ)
   (if Δ is normal)        ↓
                        Information flow
                        across token subsets
'''

In [ ]:
!git clone https://git.hodgederrick.com/hodge360/stanley-reisner-attention

In [ ]:
!cat stanley-reisner-attention/sr_attention/pyproject.toml

In [ ]:
# 1. Fix the invalid build-backend in pyproject.toml
!sed -i 's/setuptools.backends.legacy:build/setuptools.build_meta/' stanley-reisner-attention/sr_attention/pyproject.toml

# 2. Install a version of setuptools that satisfies both the project (>=68) and torch (<82)
!pip install "setuptools>=68,<82"

# 3. Install the package in editable mode
!cd stanley-reisner-attention/sr_attention && pip install -e ."[dev]" --no-build-isolation

In [2]:
import sys
import os

# Add the repository root to sys.path so 'import sr_attention' works internally
repo_root = os.path.abspath('stanley-reisner-attention')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)


In [3]:
from sr_attention.extraction import AttentionExtractor
from sr_attention.complex_builder import AttentionComplexBuilder
from sr_attention.sr_core import stanley_reisner_generators, primary_decomposition_from_facets
from sr_attention.persistence import betti_table_via_hochster, format_betti_table

# 1. Load model and extract attention patterns
extractor = AttentionExtractor("Qwen/Qwen2.5-0.5B")
cache = extractor.run("The mathematician proved the theorem using algebra.")
print(f"Sequence Length: {cache.seq_len}")

# 2. Build the filtered simplicial complex for layer 0, head 0
attn = cache.head(layer=8, head=5)
builder = AttentionComplexBuilder()
st = builder.build(attn, token_labels=cache.tokens)

# 3. Extract SR ideal generators
gens, monomials = stanley_reisner_generators(st, token_labels=cache.tokens, verbose=True)

# 4. Betti table (Topological analysis)
betti = betti_table_via_hochster(st)
print("\nBetti Table:")
print(format_betti_table(betti))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
Sequence Length: 10
SR ideal I(Δ) at τ=0.010 (t=0.990): 53 generators
  The· the
  The· theorem
  The· algebra
   mathematic· the
   mathematic· theorem
   mathematic· using
   mathematic· algebra
  ian· theorem
  ian· algebra
   proved· algebra
   the· algebra
   theorem·.
  <|endoftext|>·The· mathematic·ian
  <|endoftext|>·The· mathematic· proved
  <|endoftext|>·The· mathematic·.
  … and 38 more

Betti Table:
i\j	0	1	2	3	4	5	6	7	8
0	1	0	0	0	0	0	0	0	0
1	0	0	0	0	0	0	0	0	0
2	0	0	0	23	0	0	0	0	0
3	0	0	0	0	24	5	0	0	0
4	0	0	0	0	0	9	9	0	0
5	0	0	0	0	0	0	1	5	0
6	0	0	0	0	0	0	0	0	1


In [5]:
# Next code cell: Extract facets and verify the "two disjoint cliques" hypothesis
import gudhi

# Extract all simplices and their filtration values
simplices = list(st.get_simplices())

# Get the dimension of the complex
max_dim = st.dimension()
print(f"Complex dimension: {max_dim}")

# Fix: Calculate number of simplices per dimension manually
dim_counts = {d: 0 for d in range(max_dim + 1)}
for simplex, filtration in simplices:
    dim = len(simplex) - 1
    dim_counts[dim] += 1

for d, count in dim_counts.items():
    print(f"Simplices of dimension {d}: {count}")

# Total simplices
total = st.num_simplices()
print(f"Total simplices: {total}")

# Get all vertices
vertices = list(st.get_skeleton(0))
print(f"\nVertices: {len(vertices)}")

# Get the 1-skeleton (edges) to see the graph structure
edges = list(st.get_skeleton(1))
# Show edges with non-zero filtration (i.e., present in the complex)
edges_present = [(e, f) for e, f in edges if len(e) == 2]
print(f"Present edges: {len(edges_present)}")
for e, f in edges_present[:20]:
    print(f"  {e} -> filtration {f}")

Complex dimension: 1
Simplices of dimension 0: 10
Simplices of dimension 1: 1
Total simplices: 11

Vertices: 10
Present edges: 1
  [0, 1] -> filtration 0.6531727313995361


In [6]:
# Next code cell: Extract facets and analyze the complex structure for Layer 8 Head 3
# We use the st object directly (already has gudhi SimplexTree methods)

# 1. Basic complex statistics
print("=" * 60)
print("LAYER 8 HEAD 3 - COMPLEX STRUCTURE ANALYSIS")
print("=" * 60)

max_dim = st.dimension()
print(f"\nComplex dimension: {max_dim}")

# Count simplices by dimension using the prune trick from gudhi docs
# or just iterate through get_simplices()
from collections import Counter

dim_counts = Counter()
all_simplices = []
for simplex, filtration in st.get_simplices():
    d = len(simplex) - 1
    dim_counts[d] += 1
    all_simplices.append((simplex, filtration, d))

for d in sorted(dim_counts.keys()):
    print(f"Simplices of dimension {d}: {dim_counts[d]}")

print(f"Total simplices: {sum(dim_counts.values())}")
print(f"Number of vertices: {st.num_vertices()}")

# 2. Extract the 1-skeleton (edges) to see the graph structure
print("\n" + "=" * 60)
print("1-SKELETON (EDGES) ANALYSIS")
print("=" * 60)

edges = []
vertices = set()
for simplex, filtration in st.get_skeleton(1):
    if len(simplex) == 2:
        edges.append((simplex, filtration))
        vertices.update(simplex)

# Map vertex indices to token labels
vertex_to_token = {}
# We need to figure out the vertex ordering - let's get all 0-simplices
vertex_list = []
for simplex, filtration in st.get_skeleton(0):
    vertex_list.append((simplex[0], filtration))

# Sort by vertex index to get the order
vertex_list.sort(key=lambda x: x[0])
print(f"\nVertices (index -> token):")
for idx, (v, f) in enumerate(vertex_list):
    # The token labels should correspond to cache.tokens
    token = cache.tokens[idx] if idx < len(cache.tokens) else f"vertex_{v}"
    vertex_to_token[v] = token
    print(f"  {v}: {token} (filtration={f:.4f})")

print(f"\nEdges present in complex: {len(edges)}")
print("Edges (non-attending pairs are the complement):")
for (u, v), filtration in edges[:30]:
    print(f"  {vertex_to_token.get(u, u)} -- {vertex_to_token.get(v, v)}  (filtration={filtration:.4f})")

# 3. Compute facets (maximal simplices) by finding simplices with no cofaces
print("\n" + "=" * 60)
print("FACETS (MAXIMAL SIMPLICES)")
print("=" * 60)

# A facet is a simplex that is not a face of any higher-dimensional simplex
# We can find this by checking which simplices have no proper cofaces
facets = []
for simplex, filtration, d in all_simplices:
    # Get cofaces of codimension 1 (immediate supersets)
    cofaces = list(st.get_cofaces(simplex, 1))
    if len(cofaces) == 0:
        # This is a maximal simplex (facet)
        facets.append((simplex, filtration, d))

print(f"Number of facets: {len(facets)}")
print("\nFacets by dimension:")
facet_by_dim = Counter()
for simplex, filtration, d in facets:
    facet_by_dim[d] += 1

for d in sorted(facet_by_dim.keys()):
    print(f"  Dimension {d}: {facet_by_dim[d]} facets")

print("\nAll facets:")
for simplex, filtration, d in sorted(facets, key=lambda x: (-x[2], x[0])):
    tokens = [vertex_to_token.get(v, str(v)) for v in simplex]
    print(f"  dim {d}: {tokens} (filtration={filtration:.4f})")

# 4. Verify the "two disjoint cliques" hypothesis
print("\n" + "=" * 60)
print("VERIFYING 'TWO DISJOINT CLIQUES' HYPOTHESIS")
print("=" * 60)

# Check if the complex is the union of two cliques with no edges between them
# The generators tell us which edges are MISSING
# If hypothesis is correct: no edges between {BOS, The} and the rest

# Get all vertices
all_vertices = [v for v, _ in vertex_list]

# Check for cross edges
prefix_vertices = []  # Will hold indices of <|endoftext|> and The
content_vertices = []  # Will hold the rest

for v, _ in vertex_list:
    token = vertex_to_token.get(v, "")
    if token in ["<|<|endoftext|>", "The"]:
        prefix_vertices.append(v)
    else:
        content_vertices.append(v)

print(f"Prefix vertices: {prefix_vertices} -> {[vertex_to_token[v] for v in prefix_vertices]}")
print(f"Content vertices: {content_vertices} -> {[vertex_to_token[v] for v in content_vertices]}")

# Check if any edge connects prefix to content
cross_edges = []
within_prefix = []
within_content = []

for (u, v), filtration in edges:
    u_in_prefix = u in prefix_vertices
    v_in_prefix = v in prefix_vertices
    if u_in_prefix and v_in_prefix:
        within_prefix.append((u, v))
    elif not u_in_prefix and not v_in_prefix:
        within_content.append((u, v))
    else:
        cross_edges.append((u, v))

print(f"\nEdges within prefix: {len(within_prefix)}")
print(f"Edges within content: {len(within_content)}")
print(f"Cross edges (prefix <-> content): {len(cross_edges)}")

if len(cross_edges) == 0:
    print("\n✓ CONFIRMED: No cross edges! Complex is disjoint union of two components.")
    print(f"  Prefix component: {len(prefix_vertices)} vertices, {len(within_prefix)} edges")
    print(f"  Content component: {len(content_vertices)} vertices, {len(within_content)} edges")

    # Check if each component is a clique
    prefix_possible = len(prefix_vertices) * (len(prefix_vertices) - 1) // 2
    content_possible = len(content_vertices) * (len(content_vertices) - 1) // 2

    print(f"\n  Prefix: {len(within_prefix)}/{prefix_possible} possible edges")
    print(f"  Content: {len(within_content)}/{content_possible} possible edges")

    if len(within_prefix) == prefix_possible:
        print("  ✓ Prefix is a COMPLETE CLIQUE")
    else:
        print("  ✗ Prefix is NOT a clique")

    if len(within_content) == content_possible:
        print("  ✓ Content is a COMPLETE CLIQUE")
    else:
        print("  ✗ Content is NOT a clique")
else:
    print(f"\n✗ REJECTED: Found {len(cross_edges)} cross edges")
    for u, v in cross_edges[:10]:
        print(f"  {vertex_to_token.get(u, u)} -- {vertex_to_token.get(v, v)}")

# 5. Check higher-dimensional structure within content
print("\n" + "=" * 60)
print("HIGHER-DIMENSIONAL STRUCTURE IN CONTENT COMPONENT")
print("=" * 60)

# Get all simplices that are entirely within content vertices
content_simplices = []
for simplex, filtration, d in all_simplices:
    if all(v in content_vertices for v in simplex):
        content_simplices.append((simplex, filtration, d))

content_dim_counts = Counter()
for _, _, d in content_simplices:
    content_dim_counts[d] += 1

print("Simplices entirely within content component:")
for d in sorted(content_dim_counts.keys()):
    print(f"  Dimension {d}: {content_dim_counts[d]}")

# For a clique on 8 vertices, we expect:
# dim 0: 8, dim 1: 28, dim 2: 56, dim 3: 70, dim 4: 56, dim 5: 28, dim 6: 8, dim 7: 1
expected = {0: 8, 1: 28, 2: 56, 3: 70, 4: 56, 5: 28, 6: 8, 7: 1}
print(f"\nExpected for 8-clique: {expected}")
print(f"Actual: {dict(content_dim_counts)}")

# 6. Primary decomposition (if available from sr_core)
print("\n" + "=" * 60)
print("PRIMARY DECOMPOSITION")
print("=" * 60)

try:
    # This requires the facets to be computed
    from sr_attention.sr_core import primary_decomposition_from_facets
    # We need to format facets properly for this function
    # It likely expects a list of monomials or facet indices
    print("Facets for primary decomposition:")
    facet_monomials = []
    for simplex, filtration, d in facets:
        monomial = "*".join([vertex_to_token.get(v, str(v)) for v in simplex])
        facet_monomials.append(monomial)
        print(f"  {monomial}")

    # Note: primary_decomposition_from_facets may need specific format
    # Let's just print what we have
except Exception as e:
    print(f"Primary decomposition skipped: {e}")

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Layer 8 Head 3 builds a complex with {len(facets)} facets.")
print(f"The complex is {'DISJOINT' if len(cross_edges) == 0 else 'CONNECTED'}.")
if len(cross_edges) == 0:
    print(f"Components: {len(prefix_vertices)}-clique + {len(content_vertices)}-clique")
    print(f"This explains the symmetric Betti table: β_{{i,i+1}} = 8×C(7,i-2)")

LAYER 8 HEAD 3 - COMPLEX STRUCTURE ANALYSIS

Complex dimension: 1
Simplices of dimension 0: 10
Simplices of dimension 1: 1
Total simplices: 11
Number of vertices: 10

1-SKELETON (EDGES) ANALYSIS

Vertices (index -> token):
  0: <|endoftext|> (filtration=0.0000)
  1: The (filtration=0.0000)
  2:  mathematic (filtration=0.0000)
  3: ian (filtration=0.0000)
  4:  proved (filtration=0.0000)
  5:  the (filtration=0.0000)
  6:  theorem (filtration=0.0000)
  7:  using (filtration=0.0000)
  8:  algebra (filtration=0.0000)
  9: . (filtration=0.0000)

Edges present in complex: 1
Edges (non-attending pairs are the complement):
  <|endoftext|> -- The  (filtration=0.6532)

FACETS (MAXIMAL SIMPLICES)
Number of facets: 9

Facets by dimension:
  Dimension 0: 8 facets
  Dimension 1: 1 facets

All facets:
  dim 1: ['<|endoftext|>', 'The'] (filtration=0.6532)
  dim 0: [' mathematic'] (filtration=0.0000)
  dim 0: ['ian'] (filtration=0.0000)
  dim 0: [' proved'] (filtration=0.0000)
  dim 0: [' the'] (filtr